In [1]:
# import cupy as cp
import numpy as cp
from matplotlib import pyplot as plt
import os

def plot_density_unnormalized_int (grid, data, sum_axes=("z",), data_labels=("n"), title = "n", save_name=None, save_folder=None):
    if not isinstance(data,list):
        data = [data]
    sum_mapping = {"x": 1, "y": 0, "z": 2}
    sum_mapping_inv = {1: "x", 0: "y", 2: "z"}
    sum_ax_inds = tuple(sum_mapping[ax] for ax in sum_axes)
    data_int = []
    for d in data:
        data_int.append(cp.sum(d, axis=sum_ax_inds))
    grid_mapping = {"x":0, "y":1, "z":2}
    plot_axes = [sum_mapping_inv[ax_ind] for ax_ind in range(3) if sum_mapping_inv[ax_ind] not in sum_axes]
    plot_ax_inds = [grid_mapping[ax] for ax in plot_axes]
    fig, ax = plt.subplots()
    if len(plot_ax_inds) == 1:
        coords = cp.ravel(grid[plot_ax_inds[0]])
        for i,d in enumerate(data_int):
            ax.plot(coords,d,label=data_labels[i])
        ax.set_xlabel(plot_axes[0])
        ax.set_title(title)
        ax.legend()
        if save_name and save_folder:
            if not os.path.exists(save_folder):
                os.makedirs(save_folder)
            full_path = os.path.join(save_folder, save_name)
            fig.savefig(full_path)
    elif len(plot_ax_inds) == 2:
        coords_x = cp.ravel(grid[plot_ax_inds[0]])
        coords_y = cp.ravel(grid[plot_ax_inds[1]])
        for i,d in enumerate(data_int):
            ax.pcolormesh(coords_x, coords_y,d.T,label=data_labels[i])
        ax.set_xlabel(plot_axes[0])
        ax.set_ylabel(plot_axes[1])
        ax.set_title(title)
        ax.legend()
        if save_name and save_folder:
            if not os.path.exists(save_folder):
                os.makedirs(save_folder)
            full_path = os.path.join(save_folder, save_name)
            fig.savefig(full_path)
    else:
        raise ValueError("can only plot up to 2D")
    return fig

In [3]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.optimize import minimize_scalar
from scipy import fft as fftw
import numpy as cp

# functions
def run_simulation(grid_shape, grid_boundaries, potential_fn, dt, iterations, guess, iter_check=100, tolerance=1e-4,use_tolerance=True, save_name=None, print_all=True):
    grid,kgrid,psi = initialize_simulation(grid_shape, grid_boundaries, guess)
    errs = []
    mus = []
    psi_dev = []
    iters = []
    converged = False
    max_iter=iterations-1
    for i in range(iterations):
        psi_old = psi
        psi, psi_norm = imaginary_time_propagation_step(psi_old,grid,kgrid,potential_fn,dt)
        err = cp.sum(cp.abs(cp.abs(psi)-cp.abs(psi_old)))/cp.sum(cp.abs(psi))
        errs.append(float(err))
        if np.isnan(err):
            converged = False
            max_iter=i
            break
        mu_check = -1/2*np.log(psi_norm)/dt
        mu_dev = tolerance*2
        if len(mus)>1: mu_dev = np.abs((mu_check-mus[-2])/mu_check)
        if i % iter_check == 0 and i>2:
            print("iteration: ", i)
            print(r"$\Delta \mu/\mu$ =", mu_dev)
        mus.append(float(mu_check))
        if len(mus)>10 and mu_check<0.5*np.mean(mus[1:-1]) or mu_check<0:
            converged = False
            print("failed to converge.")
            max_iter = i
            break
        if mu_dev < tolerance and use_tolerance:
            converged = True
            print("converged!")
            max_iter = i
            break
    if save_name:
        filename = save_name+".cpz"
        cp.savez(filename, psi=psi, err=cp.array(errs))
    return grid, psi, max_iter, errs, mus, converged

def initialize_simulation(grid_shape, grid_boundaries, guess, plot_guess=False):
    xmin, xmax, ymin, ymax, zmin, zmax = grid_boundaries
    x = cp.linspace(xmin,xmax,num=grid_shape[0])
    y =  cp.linspace(ymin,ymax,num=grid_shape[1])
    z =  cp.linspace(zmin,zmax,num=grid_shape[2])
    Nx, Ny, Nz = grid_shape
    kx = 2*cp.pi*cp.fft.fftfreq(Nx)*Nx/(xmax-xmin)
    ky =  2*cp.pi*cp.fft.fftfreq(Ny)*Ny/(ymax-ymin)
    kz =  2*cp.pi*cp.fft.fftfreq(Nz)*Nz/(zmax-zmin)
    grid = cp.meshgrid(x,y,z,sparse=True)
    kgrid = cp.meshgrid(kx,ky,kz,sparse=True)
    psi = guess
    if plot_guess:
        print("Guess:")
        plot_density_unnormalized_int(grid,(cp.abs(psi)**2))
        plt.show()
    print("simulation initialized")
    return (grid,kgrid,psi)

def imaginary_time_propagation_step(psi, grid, kgrid, potential, dt):
    x, y, z = grid
    Nx, Ny, Nz = cp.size(x), cp.size(y), cp.size(z)
    Lx, Ly, Lz = cp.max(x)-cp.min(x), cp.max(y)-cp.min(y), cp.max(z)-cp.min(z)
    dLx, dLy, dLz = Lx/Nx, Ly/Ny, Lz/Nz
    psi = kinetic_propagator(psi, grid, kgrid, dt/2)
    psi = cp.exp(-dt*potential(psi, grid, kgrid))*psi
    psi = kinetic_propagator(psi, grid, kgrid, dt/2)
    psi_norm = cp.sum(cp.abs(psi)**2*dLx*dLy*dLz)
    return psi/cp.sqrt(psi_norm), psi_norm

def real_time_propagation_step(psi, grid, kgrid, potential_propagator, dt):
    return imaginary_time_propagation_step(psi, grid, kgrid, potential_propagator, 1j*dt)

def harmonic_potential(psi,grid,kgrid,gamma,anisotropy):
    x, y, z = grid
    return 1/2*(anisotropy[0]*x**2+anisotropy[1]*y**2+anisotropy[2]*z**2)+gamma*cp.abs(psi)**2

def dipolar_potential(psi,grid,kgrid,gamma,anisotropy,dipole_strength,dipole_direction):
    x, y, z = grid
    harmonic_plus_contact = 1/2*(anisotropy[0]*x**2+anisotropy[1]*y**2+anisotropy[2]*z**2)+gamma*cp.abs(psi)**2
    # dipolar term
    fft_n = fftw.fftn(cp.abs(psi)**2)
    kx, ky, kz = kgrid
    kabs = cp.sqrt(kx**2+ky**2+kz**2)
    costheta = (kx*dipole_direction[0]+ky*dipole_direction[1]+kz*dipole_direction[2])/kabs
    dipole_fft = dipole_strength*(3*costheta**2-1)
    dipole_fft[kabs==0] = 0
    dipolar = fftw.ifftn(dipole_fft*fft_n)
    return dipolar+harmonic_plus_contact


def cylindrical_dipolar_potential(psi,grid,kgrid,gamma,trap_ratio,dipole_strength):
    x, y, z = grid
    # harmonic potential
    harmonic_plus_contact = 1/2*(x**2+y**2+trap_ratio**2*z**2)+gamma*cp.abs(psi)**2
    # dipolar potential
    fft_n = fftw.fftn(cp.abs(psi)**2)
    kx, ky, kz = kgrid
    kabs = cp.sqrt(kx**2+ky**2+kz**2)
    costheta = (kz)/kabs
    costheta = cp.nan_to_num(costheta)
    dipole_fft = dipole_strength*(3*costheta**2-1)
    dipole_fft[kabs==0] = 0
    dipolar = fftw.ifftn(dipole_fft*fft_n)
    return dipolar+harmonic_plus_contact



def kinetic_propagator(psi, grid, kgrid, dt):
    fft_psi = cp.fft.fftn(psi)
    kx, ky, kz = kgrid
    ksquared = kx**2+ky**2+kz**2
    return cp.fft.ifftn(cp.exp(-1/2*ksquared*dt)*fft_psi)


def Gaussian_ansatz(grid_shape, grid_boundaries, anisotropy):
    xmin, xmax, ymin, ymax, zmin, zmax = grid_boundaries
    x = cp.linspace(xmin,xmax,num=grid_shape[0])
    y =  cp.linspace(ymin,ymax,num=grid_shape[1])
    z =  cp.linspace(zmin,zmax,num=grid_shape[2])
    grid = cp.meshgrid(x,y,z,sparse=True)
    x, y, z = grid
    Nx, Ny, Nz = cp.size(x), cp.size(y), cp.size(z)
    Lx, Ly, Lz = cp.max(x)-cp.min(x), cp.max(y)-cp.min(y), cp.max(z)-cp.min(z)
    psi = cp.exp(-(cp.sqrt(anisotropy[0])*x**2+cp.sqrt(anisotropy[1])*y**2+cp.sqrt(anisotropy[2])*z**2)/2)
    dLx, dLy, dLz = Lx/Nx, Ly/Ny, Lz/Nz
    psi_norm = cp.sum(cp.abs(psi)**2*dLx*dLy*dLz)
    return psi/cp.sqrt(psi_norm)

def Gaussian_ansatz_TF(grid_shape, grid_boundaries, gamma,anisotropy):
    xmin, xmax, ymin, ymax, zmin, zmax = grid_boundaries
    x = cp.linspace(xmin,xmax,num=grid_shape[0])
    y =  cp.linspace(ymin,ymax,num=grid_shape[1])
    z =  cp.linspace(zmin,zmax,num=grid_shape[2])
    grid = cp.meshgrid(x,y,z,sparse=True)
    x, y, z = grid
    Nx, Ny, Nz = cp.size(x), cp.size(y), cp.size(z)
    Lx, Ly, Lz = cp.max(x)-cp.min(x), cp.max(y)-cp.min(y), cp.max(z)-cp.min(z)
    mu = (15*gamma/(8*2**(3/2)*cp.pi)*cp.sqrt(anisotropy[0]*anisotropy[1]*anisotropy[2]))**(2/5)
    Rx, Ry, Rz = cp.sqrt(2*mu/anisotropy[0]), cp.sqrt(2*mu/anisotropy[1]), cp.sqrt(2*mu/anisotropy[2])
    psi = cp.exp(-2*(x**2/Rx**2+y**2/Ry**2+z**2/Rz**2))
    dLx, dLy, dLz = Lx/Nx, Ly/Ny, Lz/Nz
    psi_norm = cp.sum(cp.abs(psi)**2*dLx*dLy*dLz)
    return psi/cp.sqrt(psi_norm)



def thomas_fermi_density(grid_shape, grid_boundaries, gamma,anisotropy):
    xmin, xmax, ymin, ymax, zmin, zmax = grid_boundaries
    x = cp.linspace(xmin,xmax,num=grid_shape[0])
    y =  cp.linspace(ymin,ymax,num=grid_shape[1])
    z =  cp.linspace(zmin,zmax,num=grid_shape[2])
    Nx, Ny, Nz = grid_shape
    Lx, Ly, Lz = cp.max(x)-cp.min(x), cp.max(y)-cp.min(y),  cp.max(z)-cp.min(z)
    dLx, dLy, dLz = Lx/Nx, Ly/Ny, Lz/Nz
    x,y,z = cp.meshgrid(x,y,z,sparse=True)
    potential = 1/2*(anisotropy[0]*x**2+anisotropy[1]*y**2+anisotropy[2]*z**2)
    mu = (15*gamma/(8*2**(3/2)*cp.pi)*cp.sqrt(anisotropy[0]*anisotropy[1]*anisotropy[2]))**(2/5)
    tf = (mu-potential)/gamma
    tf[tf<0] = 0
    tf_norm = cp.sum(tf*dLx*dLy*dLz)
    return tf/tf_norm

def thomas_fermi_cyl_dipole (grid_shape, grid_boundaries, epsdd, gamma,trap_ratio,return_mu=False,return_R=False):
    def fdip(kappa):
        if kappa**2 == 1:
            return 0
        if kappa**2 > 1:
            return (1+2*kappa**2)/(1-kappa**2)+3*kappa**2*np.atan(cp.sqrt(kappa**2-1))/(kappa**2-1)**(3/2)
        return (1+2*kappa**2)/(1-kappa**2)-3*kappa**2*np.atanh(cp.sqrt(1-kappa**2))/(1-kappa**2)**(3/2)
    def aspect_ratio(edd, trap_ratio):
        fn = lambda kappa: cp.asnumpy(3*kappa**2*edd*((trap_ratio**2/2+1)*fdip(kappa)/(1-kappa**2)-1)+(edd-1)*(kappa**2-trap_ratio**2))
        fn2 = lambda kappa: cp.asnumpy(fn(kappa)**2)
        sol = minimize_scalar(fn2,bracket=(0.1,2),bounds=(0,50),method="bounded",tol=1e-9)
        return sol.x
    # calculate TF radii
    kappa = aspect_ratio(float(epsdd), float(trap_ratio))
    if kappa == 1:
        R_rho = (15/(4*cp.pi)*kappa*gamma*(1+epsdd*(3/2*kappa**2*(-2/5) - 1)))**(1/5)
    else:
        R_rho = (15/(4*cp.pi)*kappa*gamma*(1+epsdd*(3/2*kappa**2*fdip(kappa)/(1-kappa**2) - 1)))**(1/5)
    R_z = R_rho/kappa
    # calculate density profile
    xmin, xmax, ymin, ymax, zmin, zmax = grid_boundaries
    x = cp.linspace(xmin,xmax,num=grid_shape[0])
    y =  cp.linspace(ymin,ymax,num=grid_shape[1])
    z =  cp.linspace(zmin,zmax,num=grid_shape[2])
    Nx, Ny, Nz = grid_shape
    Lx, Ly, Lz = cp.max(x)-cp.min(x), cp.max(y)-cp.min(y),  cp.max(z)-cp.min(z)
    dLx, dLy, dLz = Lx/Nx, Ly/Ny, Lz/Nz
    x,y,z = cp.meshgrid(x,y,z,sparse=True)
    tf = (1-(x**2+y**2)/R_rho**2-z**2/R_z**2)
    tf[tf<0] = 0
    tf_norm = cp.sum(tf*dLx*dLy*dLz)
    if return_mu:
        mu = 15/(8*cp.pi*R_rho**2*R_z)*gamma*(1-epsdd*fdip(kappa))
        return mu
    if return_R:
        return R_rho, R_z
    return tf/tf_norm

In [4]:
# import cupy as cp
import numpy as cp
import numpy as np
import sys
import csv

# physical constants
hbar = 1.05e-34 # J.s
a0 = 5.29e-11 # m

# experimental parameters
add = 65.5*a0
m = 166*1.66e-27
gdd = 4*cp.pi*hbar**2*add/m
w_avg = 2*cp.pi*100

# simulation parameters
Nx = 2**8
Ny = 2**8
Nz = 2**8
grid_shape = [Nx, Ny, Nz]
tolerance = 1e-6
Lx = 5
Ly = 5
Lz = 5
dt = 1e-1

# quantities being scanned
trap_ratios = cp.logspace(cp.log10(0.4), cp.log10(7), 8)
ascats = cp.linspace(0.1*a0, add, 100)
atom_nums = cp.array([1e5])

# outputs
acrit_array = []
figs = []

# scan over atom number and trap ratio and find acrit
for i, atom_num in enumerate(atom_nums):
    acrit_vs_trap_ratio = []
    for trap_ratio in trap_ratios:
        wx =  w_avg/trap_ratio**(1/3)
        l = cp.sqrt(hbar/(m*wx))
        # do a binary search to find the critical ascat
        stable_vals = []
        done = False
        checking_list = ascats
        while len(checking_list) > 0:
            # check middle of list
            index = len(checking_list)//2
            ascat = checking_list[index]
            g =  4*cp.pi*hbar**2*ascat/m
            epsdd = add/ascat
            gamma = atom_num*g/(hbar*wx*l**3)
            dipole_strength = gdd*atom_num/(hbar*wx*l**3)
            anisotropy = [1,1,trap_ratio**2]
            # simulation parameters
            mu_TF =(15*gamma/(8*2**(3/2)*np.pi)*np.sqrt(anisotropy[0]*anisotropy[1]*anisotropy[2]))**(2/5)
            mu_TF_dip =(15*dipole_strength/(8*2**(3/2)*np.pi)*np.sqrt(anisotropy[0]*anisotropy[1]*anisotropy[2]))**(2/5)
            time_step = min([dt/mu_TF,dt/mu_TF_dip])
            R_TF = np.sqrt(2*mu_TF)
            R_z_TF = R_TF/trap_ratio
            grid_boundaries = [-Lx/2*R_TF, Lx/2*R_TF, -Ly/2*R_TF, Ly/2*R_TF, -Lz/2*R_z_TF, Lz/2*R_z_TF]
            print("ascat:", ascat/a0, "trap ratio:", trap_ratio, "atom number:", atom_num)
            iterations = 100000
            potential_fn = lambda psi, grid, kgrid: cylindrical_dipolar_potential(psi,grid,kgrid,gamma,trap_ratio,dipole_strength)
            guess = Gaussian_ansatz_TF(grid_shape, grid_boundaries,cp.abs(gamma),anisotropy)
            grid, psi, max_iter, err, mu, conv = run_simulation(grid_shape,grid_boundaries,potential_fn,time_step,iterations,guess,tolerance=tolerance,iter_check=50,print_all=True)
            # if stable, check lower for unstable
            n_sim = np.abs(psi)**2
            tf_n = thomas_fermi_density(grid_shape, grid_boundaries, gamma, anisotropy)
            tfd_n = thomas_fermi_cyl_dipole (grid_shape, grid_boundaries, epsdd, gamma, trap_ratio)
            plot_density_unnormalized_int(grid,[n_sim,tf_n,tfd_n],sum_axes=("y","z"), data_labels=("Sim", "TF", "DTF"))
            plot_density_unnormalized_int(grid,[n_sim],sum_axes=("y"), data_labels=("Sim"))
            plt.show()
            iters = np.arange(0,max_iter+1,1)
            plt.scatter(iters[1:], mu[1:])
            plt.title(f"$\mu$")
            plt.show()
            plt.title(f"error")
            plt.scatter(iters, err)
            plt.yscale("log")
            plt.show()
            if conv:
              checking_list = checking_list[:len(checking_list)//2]
              stable_vals.append(float(ascat))
            # if unstable, check upper for stable
            else:
              checking_list = checking_list[len(checking_list)//2+1:]
        acrit_vs_trap_ratio.append(min(stable_vals))
    acrit_array.append(acrit_vs_trap_ratio)

print(repr(np.array(acrit_array)/a0))

<>:77: SyntaxWarning: invalid escape sequence '\m'
<>:77: SyntaxWarning: invalid escape sequence '\m'
/var/folders/m7/bs8_zgtx3qg7d726x69dyt3c0000gn/T/ipykernel_39489/4116649700.py:77: SyntaxWarning: invalid escape sequence '\m'
  plt.title(f"$\mu$")


ascat: 33.13030303030303 trap ratio: 0.4 atom number: 100000.0
simulation initialized


/var/folders/m7/bs8_zgtx3qg7d726x69dyt3c0000gn/T/ipykernel_39489/660661304.py:106: RuntimeWarning: invalid value encountered in divide
  costheta = (kz)/kabs
/var/folders/m7/bs8_zgtx3qg7d726x69dyt3c0000gn/T/ipykernel_39489/4116649700.py:77: SyntaxWarning: invalid escape sequence '\m'
  plt.title(f"$\mu$")


KeyboardInterrupt: 